In [ ]:
# BLOCK 0  – GLOBAL SETUP: imports ▸ paths ▸ flags ▸ constants
# ════════════════════════════════════════════════════════════════════════════════

# --- core libs
import numpy as np
import nibabel as nib
import pandas as pd
import gc                            # ← added
import torch                         # ← added

# --- imaging / DL
import cv2, matplotlib.pyplot as plt
from ultralytics import YOLO
from totalsegmentator.python_api import totalsegmentator

# --- misc libs
from pathlib import Path
from collections import Counter

# ── PATHS (EDIT HERE) ───────────────────────────────────────────────────────────
MODEL_PATH = Path(r"C:\Users\Ryan Krishna\Documents\Overscanning\yolo_runs\yolo11_pubic_symphysis_m_hardtrain\weights\best.pt")

NIFTI_DIR  = Path(r"D:\Abdomen_CT_Bone_Mets_Nifti")
CSV_PATH   = NIFTI_DIR / "overscanning_results.csv"

# ── FLAGS ───────────────────────────────────────────────────────────────────────
DISPLAY_DETECTION = True     # draw green box on best slice
FAST_MODEL        = False    # TotalSegmentator "fast" mode
MULTI_LABEL_MASK  = True     # 1 = liver, 2 = spleen

# ── CONSTANTS ───────────────────────────────────────────────────────────────────
FINAL_CONF    = 0.20     # YOLO confidence threshold
BACKGROUND_HU = -300     # HU ≤ –300 → treated as air/outside body
model         = YOLO(str(MODEL_PATH))   # load once, reused everywhere

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
#  Pubic-symphysis detection ➔ caudal-overscan  (robust, femur-aware)
# ════════════════════════════════════════════════════════════════════════════════
import traceback, gc, torch, pandas as pd, numpy as np, nibabel as nib, cv2
from pathlib import Path
from totalsegmentator.python_api import totalsegmentator

# ────────────────────────────────────────────────────────────────────────────────
# 0)  SCANS ALREADY IN CSV
# ────────────────────────────────────────────────────────────────────────────────
if CSV_PATH.exists():
    done_df  = pd.read_csv(CSV_PATH)
    done_set = set(done_df["file_name"].tolist())
    print(f"↪️  {len(done_set)} rows already in CSV – they’ll be skipped\n")
else:
    done_set = set()

# ────────────────────────────────────────────────────────────────────────────────
# 1)  HELPER FUNCTIONS
# ────────────────────────────────────────────────────────────────────────────────
def preprocess_slice(arr: np.ndarray) -> np.ndarray:
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    arr = (arr * 255).astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)


def ensure_femur_mask(ct_path: Path) -> Path | None:
    """
    Creates / returns a merged femur mask (label 1). Returns None on TS failure.
    """
    out_dir     = ct_path.parent / "ts_femur"
    fem_l_path  = out_dir / "femur_left.nii.gz"
    fem_r_path  = out_dir / "femur_right.nii.gz"
    merged_path = ct_path.parent / "femur_combined.nii.gz"

    if merged_path.exists():
        return merged_path

    # ── TotalSegmentator with GPU → CPU fallback ────────────────────────────
    if not (fem_l_path.exists() and fem_r_path.exists()):
        out_dir.mkdir(exist_ok=True)
        for dev in ("gpu", "cpu"):
            try:
                totalsegmentator(
                    ct_path, out_dir,
                    roi_subset=["femur_left", "femur_right"],
                    task="total",
                    fast=FAST_MODEL,
                    device=dev,
                )
                break                              # success
            except Exception as e:
                print(f"⚠️  TS({dev}) {ct_path.name}: {e}")
        else:
            return None                            # both runs failed

    # ── merge masks ─────────────────────────────────────────────────────────
    try:
        fem_l = nib.load(fem_l_path).get_fdata() > 0
        fem_r = nib.load(fem_r_path).get_fdata() > 0
    except FileNotFoundError:
        return None

    merged = (fem_l | fem_r).astype(np.uint8)
    if not merged.any():
        return None                                # no femurs in slice range

    ref = nib.load(fem_l_path if fem_l_path.exists() else fem_r_path)
    nib.save(nib.Nifti1Image(merged, ref.affine, ref.header), merged_path)

    for p in (fem_l_path, fem_r_path):  # clean up
        if p.exists():
            p.unlink()

    return merged_path


def femur_top_info(ct_path: Path) -> tuple[int, float] | None:
    """
    Returns (slice_idx, world-z_mm) of the cranial-most femur voxel,
    or None if no femurs are present / segmentation failed.
    """
    m = ensure_femur_mask(ct_path)
    if m is None:
        return None
    mask = nib.load(str(m))
    mask_np = mask.get_fdata() > 0
    slices = np.where(mask_np.any(axis=(0, 1)))[0]
    if slices.size == 0:
        return None
    affine = mask.affine
    z_coords = [(k, float((affine @ [0, 0, k, 1])[2])) for k in slices]
    return max(z_coords, key=lambda t: t[1])       # cranial-most


def find_valid_pubic_slice(ct_path: Path, z_cutoff_mm: float) -> int | None:
    """
    Returns the highest-confidence YOLO slice ≤ z_cutoff_mm, or None.
    """
    ct = nib.load(str(ct_path))
    affine = ct.affine
    vol = ct.get_fdata()
    H, W, Z = vol.shape

    best_conf, best_slice = -1.0, None
    for z in range(Z):
        if float((affine @ [0, 0, z, 1])[2]) > z_cutoff_mm:
            continue
        img = preprocess_slice(vol[:, :, z])
        res = model.predict(img, conf=FINAL_CONF, device=0, save=False)[0]
        for b in sorted(res.boxes, key=lambda bb: float(bb.conf), reverse=True):
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            if vol[int((y1+y2)/2), int((x1+x2)/2), z] <= BACKGROUND_HU:
                continue
            cx = int((x1+x2)/2)
            if abs(cx - W//2) > 0.20 * W:
                continue
            win = vol[max(0,int((y1+y2)/2)-10):min(H,int((y1+y2)/2)+10),
                      max(0,cx-10):min(W,cx+10), z]
            if win.mean() < 150:
                continue
            conf = float(b.conf)
            if conf > best_conf:
                best_conf, best_slice = conf, z
            break
    return best_slice


# ────────────────────────────────────────────────────────────────────────────────
# 2)  FILTER: keep only original CT volumes
# ────────────────────────────────────────────────────────────────────────────────
def is_ct_vol(p: Path) -> bool:
    if p.parent.name.startswith("ts_"):
        return False
    if p.name.endswith("_combined.nii.gz"):
        return False
    if p.name.startswith(("femur_", "liver_", "spleen_")):
        return False
    return True

nii_paths = [p for p in NIFTI_DIR.rglob("*.nii*") if is_ct_vol(p)]
print(f"🔎 {len(nii_paths)} CT volumes found\n")

# ────────────────────────────────────────────────────────────────────────────────
# 3)  MAIN LOOP
# ────────────────────────────────────────────────────────────────────────────────
for ct_path in nii_paths:
    if ct_path.name in done_set:
        continue
    try:
        print(f"▶ {ct_path.relative_to(NIFTI_DIR.parent)}")

        # (a) femur segmentation
        fem_data = femur_top_info(ct_path)
        if fem_data:
            fem_slice, fem_top_z = fem_data
            z_cut = fem_top_z
        else:
            fem_slice, fem_top_z = None, np.nan
            z_cut = float("inf")                   # no gating

        # (b) YOLO detection with gating
        pubic_slice = find_valid_pubic_slice(ct_path, z_cut)
        if pubic_slice is None and fem_slice is not None:
            pubic_slice   = fem_slice             # fallback
            source_label  = "FemurFallback"
        elif pubic_slice is None:                 # no femur + no YOLO
            print("   ⚠️  no pubic symphysis found – skipping")
            continue
        else:
            source_label  = "YOLO" if not np.isnan(fem_top_z) else "YOLO_NoFemur"

        # (c) caudal overscan metrics
        ct_img  = nib.load(str(ct_path))
        affine  = ct_img.affine
        Z       = ct_img.shape[2]
        pubic_z = float((affine @ [0, 0, pubic_slice, 1])[2])
        end_z   = min(float((affine @ [0, 0, k, 1])[2]) for k in range(Z))
        caudal  = abs(end_z - pubic_z)

        # (d) assemble row (locked column order)
        row = {
            "file_name"         : ct_path.name,
            "pubic_z_mm"        : int(round(pubic_z)),
            "scan_end_z_mm"     : int(round(end_z)),
            "caudal_overscan_mm": int(round(caudal)),
            "femur_top_z_mm"    : (int(round(fem_top_z))
                                   if not np.isnan(fem_top_z) else np.nan),
            "pubic_source"      : source_label,
        }

        # (e) update CSV, preserving cranial columns
        if CSV_PATH.exists():
            df = pd.read_csv(CSV_PATH)
            if row["file_name"] in df["file_name"].values:
                ix = df.index[df["file_name"] == row["file_name"]][0]
                for col in row:
                    df.at[ix, col] = row[col]
            else:
                df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
            df.sort_values("file_name").to_csv(CSV_PATH, index=False)
        else:
            pd.DataFrame([row]).to_csv(CSV_PATH, index=False)

        done_set.add(ct_path.name)
        print("   ✓ saved")

    except Exception as e:
        print("   ⚠️  unexpected error – continuing")
        traceback.print_exc(limit=1)

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\n✅ Finished. CSV now contains {len(done_set)} rows")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 3  – TotalSegmentator (liver + spleen) → combined masks
# ════════════════════════════════════════════════════════════════════════════════
def ensure_liver_spleen_mask(ct_path: Path) -> Path:
    out_dir      = ct_path.parent / "ts_liver_spleen"
    liver_mask   = out_dir / "liver.nii.gz"
    spleen_mask  = out_dir / "spleen.nii.gz"
    merged_mask  = ct_path.parent / "liver_spleen_combined.nii.gz"

    if merged_mask.exists():
        return merged_mask

    # 1) run TotalSegmentator if needed
    if not (liver_mask.exists() and spleen_mask.exists()):
        out_dir.mkdir(exist_ok=True)
        totalsegmentator(
            ct_path, out_dir,
            roi_subset=["liver", "spleen"],
            task="total",
            fast=FAST_MODEL,
            device="gpu"
        )

    # 2) merge masks
    liver_img   = nib.load(liver_mask)
    spleen_img  = nib.load(spleen_mask)
    liver_data  = liver_img.get_fdata() > 0
    spleen_data = spleen_img.get_fdata() > 0

    if MULTI_LABEL_MASK:
        combined = np.zeros(liver_data.shape, dtype=np.uint8)
        combined[liver_data]  = 1
        combined[spleen_data] = 2
    else:
        combined = (liver_data | spleen_data).astype(np.uint8)

    merged_img = nib.Nifti1Image(combined, liver_img.affine, liver_img.header)
    nib.save(merged_img, merged_mask)

    # 3) delete individual masks
    for f in (liver_mask, spleen_mask):
        if f.exists():
            f.unlink()

    return merged_mask

nii_paths = sorted(NIFTI_DIR.rglob("*.nii*"))
print(f"Found {len(nii_paths)} NIfTI files\n")

# --- run segmentation (skip if combined already present) ---
for ct_path in nii_paths:
    rel = ct_path.relative_to(NIFTI_DIR.parent)
    print(f"▶ Segmentation check: {rel}")
    ensure_liver_spleen_mask(ct_path)

print("\n✓ Liver + spleen masks ensured for all scans")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 4 – Cranial overscan  (robust, incremental CSV update)
# ════════════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np, nibabel as nib, gc, torch
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 1)  FILTER: only original CT volumes
# ────────────────────────────────────────────────────────────────────────────────
def is_ct_vol(p: Path) -> bool:
    # skip segmentation sub-folders
    if p.parent.name.startswith("ts_"):
        return False
    # skip any mask file whose OWN filename starts with ts_
    if p.name.startswith("ts_"):
        return False
    # skip merged masks
    if p.name.endswith("_combined.nii.gz"):
        return False
    # skip obvious single-organ masks
    if p.name.startswith(("femur_", "liver_", "spleen_")):
        return False
    return True

ct_paths = [p for p in NIFTI_DIR.rglob("*.nii*") if is_ct_vol(p)]
print(f"🔎 {len(ct_paths)} CT volumes for cranial-overscan pass\n")

# ────────────────────────────────────────────────────────────────────────────────
# 2)  LOAD or CREATE CSV
# ────────────────────────────────────────────────────────────────────────────────
if CSV_PATH.exists():
    csv_df = pd.read_csv(CSV_PATH)
else:
    csv_df = pd.DataFrame(columns=[
        "file_name", "pubic_z_mm", "scan_end_z_mm", "caudal_overscan_mm",
        "femur_top_z_mm", "pubic_source",
        "liver_spleen_z_mm", "scan_start_z_mm", "cranial_overscan_mm", "top_organ"
    ])

# ────────────────────────────────────────────────────────────────────────────────
# 3)  CRANIAL-OVERSCAN FUNCTION
# ────────────────────────────────────────────────────────────────────────────────
def cranial_overscan(ct_path: Path, mask_path: Path) -> tuple[int, int, int, str]:
    ct_img   = nib.load(str(ct_path))
    mask_img = nib.load(str(mask_path))
    affine   = ct_img.affine
    mask_np  = mask_img.get_fdata()

    seg_slices = np.where(mask_np.any(axis=(0, 1)))[0]
    if seg_slices.size == 0:
        raise RuntimeError("combined mask empty")

    z_coords = [(k, float((affine @ [0, 0, k, 1])[2])) for k in seg_slices]

    Z = ct_img.shape[2]
    z_edge0 = float((affine @ [0, 0,      0, 1])[2])
    z_edgeN = float((affine @ [0, 0, Z - 1, 1])[2])
    cranial_edge_z = max(z_edge0, z_edgeN)

    highest_slice, highest_z = min(z_coords, key=lambda t: abs(t[1] - cranial_edge_z))

    labels     = mask_np[:, :, highest_slice][mask_np[:, :, highest_slice] > 0].astype(int)
    organ_map  = {1: "Liver", 2: "Spleen"}
    organ_top  = organ_map.get(int(np.bincount(labels).argmax()), "Unknown")

    cranial_mm    = int(round(abs(cranial_edge_z - highest_z)))
    scan_start_mm = int(round(cranial_edge_z))
    organ_z_mm    = int(round(highest_z))
    return cranial_mm, organ_z_mm, scan_start_mm, organ_top

# ────────────────────────────────────────────────────────────────────────────────
# 4)  MAIN LOOP – incremental CSV write
# ────────────────────────────────────────────────────────────────────────────────
for ct_path in ct_paths:
    mask_path = ct_path.parent / "liver_spleen_combined.nii.gz"
    if not mask_path.exists():
        print(f"⚠️  no liver+spleen mask for {ct_path.name} – skipping")
        continue

    try:
        cranial_mm, organ_z_mm, scan_start_mm, organ_top = cranial_overscan(ct_path, mask_path)
    except Exception as e:
        print(f"⚠️  {ct_path.name}: {e} – skipping")
        continue

    row = {
        "file_name"          : ct_path.name,
        "liver_spleen_z_mm"  : organ_z_mm,
        "scan_start_z_mm"    : scan_start_mm,
        "cranial_overscan_mm": cranial_mm,
        "top_organ"          : organ_top,
    }

    if ct_path.name in csv_df["file_name"].values:
        ix = csv_df.index[csv_df["file_name"] == ct_path.name][0]
        for k, v in row.items():
            csv_df.at[ix, k] = v
    else:
        csv_df = pd.concat([csv_df, pd.DataFrame([row])], ignore_index=True)

    csv_df.sort_values("file_name").to_csv(CSV_PATH, index=False)
    print(f"   ✓ cranial metrics saved for {ct_path.name}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n✅ Cranial pass finished → {CSV_PATH.resolve()}")

In [ ]:
# # ════════════════════════════════════════════════════════════════════════════════
# # Replace "file_name" with patient ID and delete the extra column
# # ════════════════════════════════════════════════════════════════════════════════
# import pandas as pd

# CSV_PATH = r"D:\Abdomen_CT_Bone_Mets_Nifti\overscanning_results.csv"  # adjust if needed
# DRY_RUN  = False  # ▶ False → write changes   |   True → preview only

# # ── load CSV ────────────────────────────────────────────────────────────────────
# df = pd.read_csv(CSV_PATH)

# # derive ID (first two underscore‑separated parts) and overwrite file_name
# new_ids = (
#     df["file_name"]
#       .str.replace(".nii.gz", "", regex=False)
#       .str.extract(r"^([^_]+_[0-9]+)", expand=False)
# )

# df["file_name"] = new_ids

# # drop the old helper column if it exists
# df.drop(columns=[c for c in ("patient_id",) if c in df.columns], inplace=True)

# # ── preview or save ─────────────────────────────────────────────────────────────
# if DRY_RUN:
#     print("\n🟡 DRY‑RUN: preview after replacement (first 8 rows)\n")
#     display(df.head(8))
# else:
#     df.to_csv(CSV_PATH, index=False)
#     print(f"✅ CSV updated – patient IDs now in 'file_name' at → {CSV_PATH}")